# Dimensionality reduction

This notebook will take you through the different dimensionality reduction data techniques which we went through in the Lecture.

First, lets load/install the packages that we need:

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings("ignore")

We need to define the file which contains the features that you should have collected from the last Lecture. If you didn't manage to get this working properly, the file which I collected earlier. It doesn't matter what your file is called, as long as you specify it between the quotation marks below:

In [ ]:
# features_file = ""
features_file = "./AQME_outputs/AQME-ROBERT_interpret_acid_anions_canonical.csv"

Next we need to put these features into a pandas Dataframe - which is essentially a Excel spreadsheet in Python.

In [ ]:
df = pd.read_csv(features_file, index_col=0)
df.head()

Before getting to the dimensionality reduction part, it's usually a good idea to remove features which are collinear with each other (this prevents two or more columns all providing the same or similar chemical information to the chemical space).

In [ ]:
# Remove non-numeric columns (like SMILES)
df_numeric = df.select_dtypes(include=[np.number])

# Remove rows with NaN values
df_numeric = df_numeric.dropna()

In [ ]:
# Calculate the correlation matrix
correlation_matrix = df_numeric.corr()

# Now we can do the collinearity check. We will use the correlation matrix to identify any pairs of features that are highly correlated with each other. We will then drop one of the features from each pair of highly correlated features.
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, cmap="coolwarm", center=0)
plt.title("Correlation Matrix Heatmap")
plt.tight_layout()

In [ ]:
# remove features which are highly correlated

cutoff = 0.9

print(f"Original number of features: {df_numeric.shape[1]}")
df_corr = df_numeric.corr()
df_not_correlated = ~(df_corr.mask(np.tril(np.ones([len(df_corr)]*2, dtype=bool))).abs() > cutoff).any()
un_corr_index = df_not_correlated.loc[df_not_correlated[df_not_correlated.index] == True].index
sel_features_df = df_numeric[un_corr_index]
print(f"Reduced number of features: {sel_features_df.shape[1]}")

In [ ]:
# Lets look at the remaining features.
sel_features_df.head()

Note that there is no chemical basis for removing one feature over another, they are removed based on the order they get to in the spreadsheet. Alternative approaches might be to keep the ones which are chemically meaningful to a given reaction (and then do a collinearlity removal depending on what's left).

*Based on the amide coupling reaction, which features might be the most interpretable ones to keep?*

Now we have removed the features which are collinear, we need to make sure they are all on the same scale. Don't worry too much about what this means now, we will cover this in Lecture 4.

In [ ]:
# Initialize the StandardScaler
scaler = StandardScaler()

# Fit and transform the selected features
sel_features_df_scaled = scaler.fit_transform(sel_features_df)

# Convert back to DataFrame to preserve column names and index
sel_features_df_scaled = pd.DataFrame(sel_features_df_scaled, columns=sel_features_df.columns, index=sel_features_df.index)

sel_features_df_scaled.head()

# finally save the scaled features to a new csv file (you'll need this for Exercise 1)
sel_features_df_scaled.to_csv("scaled_features.csv")

Finally, we are ready to run the PCA. First we need to define the number of components we want to have. Let's say 4 to start with.

In [ ]:
pca = PCA(n_components=4)
pca_reduced = pca.fit(sel_features_df_scaled)

We can look at the PCA components and the explained variance for each component:

In [ ]:
# PCA components
print(pca_reduced.components_)

# Explained variance ratio
print(pca_reduced.explained_variance_ratio_ * 100)

Finally, we can extract the individual principal components and plot the result.

In [ ]:
sel_features_df["PC1"] = pca_reduced.transform(sel_features_df_scaled)[:, 0]
sel_features_df["PC2"] = pca_reduced.transform(sel_features_df_scaled)[:, 1]
sel_features_df["PC3"] = pca_reduced.transform(sel_features_df_scaled)[:, 2]
sel_features_df["PC4"] = pca_reduced.transform(sel_features_df_scaled)[:, 3]

In [ ]:
dim1 = "PC1"
dim2 = "PC2"

plt.figure(figsize=(6,6))
sns.scatterplot(data=sel_features_df, x=dim1, y=dim2)
plt.title(f"PCA Scatter Plot: {dim1} vs {dim2}")
plt.xlabel(f"{dim1} ({pca_reduced.explained_variance_ratio_[0] * 100:.2f}% variance)")
plt.ylabel(f"{dim2} ({pca_reduced.explained_variance_ratio_[1] * 100:.2f}% variance)")
plt.grid()
plt.tight_layout()
plt.show()

*Sometimes it is useful to color these plots using the features - see if you can work out how to do that*

To finish off, let's export the PCA coordinates to a CSV file to use later.

In [ ]:
sel_features_df.to_csv("pca_reduced_features.csv")